# AI-Based Operational Risk Forecasting Model

## Overview
This notebook builds machine learning models to predict high-loss operational risk events using engineered features derived from the P-COLD dataset.

The primary goal is to estimate the probability of future high-loss events within defined forecasting windows and support proactive risk monitoring.

## Objectives
- Train predictive models on engineered risk features
- Handle class imbalance using threshold tuning
- Compare model performance across algorithms
- Evaluate performance across multiple forecasting horizons (90, 180, 365 days)
- Export predictions and metrics for dashboard integration

## Models Used
- Random Forest (primary model)
- Logistic Regression (baseline model)
- Extra Trees (ensemble comparison)

## Key Techniques
- Feature preprocessing with pipelines
- Probability-based prediction
- Threshold optimization (F1-based)
- Model evaluation using ROC-AUC, precision, recall, and F1 score

## Outputs
- Risk prediction dataset with probabilities and classifications
- Model performance metrics
- Multi-horizon comparison results

## Notes
The 90-day horizon serves as the primary forecasting window, while 180-day and 365-day models are used for robustness analysis and comparison.

In [1]:
# Load libraries and the primary 90-day modeling dataset.
# This notebook trains the main forecasting model, compares algorithms,
# and exports predictions/metrics for Power BI.

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import joblib
import os

def find_best_threshold(y_true, y_proba, metric="f1"):
    thresholds = np.arange(0.01, 0.991, 0.01)
    best_threshold = 0.5
    best_score = -1

    rows = []

    for t in thresholds:
        preds = (y_proba >= t).astype(int)

        precision = precision_score(y_true, preds, zero_division=0)
        recall = recall_score(y_true, preds, zero_division=0)
        f1 = f1_score(y_true, preds, zero_division=0)

        rows.append({
            "threshold": t,
            "precision": precision,
            "recall": recall,
            "f1_score": f1
        })

        if metric == "f1":
            score = f1
        elif metric == "recall":
            score = recall
        elif metric == "precision":
            score = precision
        else:
            score = f1

        if score > best_score:
            best_score = score
            best_threshold = t

    threshold_df = pd.DataFrame(rows)
    return best_threshold, threshold_df

np.random.seed(42)

df = pd.read_csv("../data/processed/future_model_dataset_90d.csv")

print("Shape:", df.shape)
display(df.head())

Shape: (2233, 50)


,business_line,event_type,causal_factor,province_occurred,year_month,event_count,total_loss_cny,avg_loss_cny,median_loss_cny,max_loss_cny,...,future_high_loss,future_any_event,business_line_freq,event_type_freq,causal_factor_freq,province_occurred_freq,month_num,quarter_num,year_num,horizon_days
0,Agency services,Business disruption and system failures,System,Guangdong,2008-02,1,0.0,0.0,0.0,0.0,...,0,0,0.00627,0.009404,0.017465,0.044783,2,1,2008,90
1,Agency services,"Clients, products & business practices",People,Beijing,2015-05,1,70000.0,70000.0,70000.0,70000.0,...,0,0,0.00627,0.142409,0.459472,0.094044,5,2,2015,90
2,Agency services,"Clients, products & business practices",People,Hainan,2009-10,1,50000.0,50000.0,50000.0,50000.0,...,0,0,0.00627,0.142409,0.459472,0.010300,10,4,2009,90
3,Agency services,"Execution, delivery & process management",External event,Shanghai,2010-08,1,6039.0,6039.0,6039.0,6039.0,...,0,0,0.00627,0.152709,0.322884,0.045231,8,3,2010,90
4,Agency services,"Execution, delivery & process management",People,Hebei,2003-07,1,1500.0,1500.0,1500.0,1500.0,...,0,0,0.00627,0.152709,0.459472,0.011196,7,3,2003,90


## Define the Target
Binary high risk

In [2]:
# Define the supervised learning target.
# future_high_loss = 1 means a high-loss operational risk event occurs
# within the selected future horizon.

target_col = "future_high_loss"

print("Available columns:")
print(df.columns.tolist())

df[target_col] = df[target_col].astype(int)

print("Target counts:")
print(df[target_col].value_counts(dropna=False))
print("\nTarget distribution:")
print(df[target_col].value_counts(normalize=True, dropna=False))

Available columns:
['business_line', 'event_type', 'causal_factor', 'province_occurred', 'year_month', 'event_count', 'total_loss_cny', 'avg_loss_cny', 'median_loss_cny', 'max_loss_cny', 'high_loss_count', 'avg_banks_involved', 'month_start', 'event_count_lag_1', 'total_loss_lag_1', 'high_loss_count_lag_1', 'event_count_lag_2', 'total_loss_lag_2', 'high_loss_count_lag_2', 'event_count_lag_3', 'total_loss_lag_3', 'high_loss_count_lag_3', 'event_count_lag_6', 'total_loss_lag_6', 'high_loss_count_lag_6', 'event_count_lag_12', 'total_loss_lag_12', 'high_loss_count_lag_12', 'event_count_roll_3', 'total_loss_roll_3', 'high_loss_roll_3', 'event_count_roll_6', 'total_loss_roll_6', 'high_loss_roll_6', 'event_count_roll_12', 'total_loss_roll_12', 'high_loss_roll_12', 'loss_per_event', 'log_total_loss', 'future_window_end', 'future_high_loss', 'future_any_event', 'business_line_freq', 'event_type_freq', 'causal_factor_freq', 'province_occurred_freq', 'month_num', 'quarter_num', 'year_num', 'horiz

## Select Features

In [3]:
# Select categorical, lagged, rolling, frequency, and time-based features.
# These features are designed to predict future high-loss risk without using future information.

feature_cols = [
    "business_line",
    "event_type",
    "causal_factor",
    "province_occurred",
    "event_count",
    "total_loss_cny",
    "avg_loss_cny",
    "median_loss_cny",
    "max_loss_cny",
    "high_loss_count",
    "avg_banks_involved",
    "event_count_lag_1",
    "event_count_lag_2",
    "event_count_lag_3",
    "event_count_lag_6",
    "event_count_lag_12",
    "total_loss_lag_1",
    "total_loss_lag_2",
    "total_loss_lag_3",
    "total_loss_lag_6",
    "total_loss_lag_12",
    "high_loss_count_lag_1",
    "high_loss_count_lag_2",
    "high_loss_count_lag_3",
    "high_loss_count_lag_6",
    "high_loss_count_lag_12",
    "event_count_roll_3",
    "event_count_roll_6",
    "event_count_roll_12",
    "total_loss_roll_3",
    "total_loss_roll_6",
    "total_loss_roll_12",
    "high_loss_roll_3",
    "high_loss_roll_6",
    "high_loss_roll_12",
    "loss_per_event",
    "log_total_loss",
    "business_line_freq",
    "event_type_freq",
    "causal_factor_freq",
    "province_occurred_freq",
    "month_num",
    "quarter_num",
    "year_num"
]

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Feature matrix shape:", X.shape)

Feature matrix shape: (2233, 44)


## Preprocessing and Model Pipeline

In [4]:
# Build a preprocessing and modeling pipeline.
# Categorical variables are imputed and one-hot encoded; numeric fields are imputed with zero.
# Random Forest is used as the primary nonlinear ensemble model.

categorical = ["business_line", "event_type", "causal_factor", "province_occurred"]
numeric = [c for c in feature_cols if c not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0))
        ]), numeric)
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", model)
])

## Train / Test Split

In [5]:
# Split the dataset into training and testing sets while preserving class balance.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate :", round(y_test.mean(), 4))

Train shape: (1786, 44)
Test shape : (447, 44)
Train positive rate: 0.0213
Test positive rate : 0.0224


## Fit Model

In [6]:
# Train the main Random Forest model and tune the probability threshold
# using F1 score to handle class imbalance.

pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
best_threshold_main, threshold_df_main = find_best_threshold(y_test, proba, metric="f1")
pred = (proba >= best_threshold_main).astype(int)

print("Best threshold for main Random Forest:", best_threshold_main)
display(threshold_df_main)

Best threshold for main Random Forest: 0.32


,threshold,precision,recall,f1_score
0,0.01,0.023095,1.0,0.045147
1,0.02,0.025316,1.0,0.049383
2,0.03,0.028818,1.0,0.056022
3,0.04,0.030508,0.9,0.059016
4,0.05,0.035573,0.9,0.068441
...,...,...,...,...
94,0.95,0.000000,0.0,0.000000
95,0.96,0.000000,0.0,0.000000
96,0.97,0.000000,0.0,0.000000
97,0.98,0.000000,0.0,0.000000


## Evaluate Model

In [7]:
# Evaluate the main model using ROC-AUC, confusion matrix, and classification metrics.
# ROC-AUC measures ranking ability, while precision/recall/F1 depend on the tuned threshold.

roc = roc_auc_score(y_test, proba)
cm = confusion_matrix(y_test, pred)
report = classification_report(y_test, pred, zero_division=0)

print("ROC-AUC:", round(roc, 4))
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

ROC-AUC: 0.7389

Confusion Matrix:
 [[416  21]
 [  7   3]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.95      0.97       437
           1       0.12      0.30      0.18        10

    accuracy                           0.94       447
   macro avg       0.55      0.63      0.57       447
weighted avg       0.96      0.94      0.95       447



## Score all rows

In [8]:
# Score every row in the 90-day dataset and create prediction flags
# using the tuned threshold from the test set.

df["future_high_loss_pred_proba"] = pipe.predict_proba(X)[:, 1]
df["future_high_loss_pred_flag"] = (df["future_high_loss_pred_proba"] >= best_threshold_main).astype(int)

display(
    df[[
        "business_line",
        "event_type",
        "causal_factor",
        "province_occurred",
        "year_month",
        "future_high_loss",
        "future_high_loss_pred_proba",
        "future_high_loss_pred_flag"
    ]].head(20)
)

,business_line,event_type,causal_factor,province_occurred,year_month,future_high_loss,future_high_loss_pred_proba,future_high_loss_pred_flag
0,Agency services,Business disruption and system failures,System,Guangdong,2008-02,0,0.032841,0
1,Agency services,"Clients, products & business practices",People,Beijing,2015-05,0,0.030564,0
2,Agency services,"Clients, products & business practices",People,Hainan,2009-10,0,0.115251,0
3,Agency services,"Execution, delivery & process management",External event,Shanghai,2010-08,0,0.115362,0
4,Agency services,"Execution, delivery & process management",People,Hebei,2003-07,0,0.055863,0
5,Agency services,"Execution, delivery & process management",People,Shandong,2006-01,0,0.097255,0
6,Agency services,"Execution, delivery & process management",Process,Shanghai,2006-04,0,0.059018,0
7,Agency services,"Execution, delivery & process management",Process,Zhejiang,2018-10,0,0.025893,0
8,Agency services,Internal fraud,People,Gansu,2007-01,0,0.062801,0
9,Agency services,Internal fraud,People,Guangdong,2004-08,0,0.049823,0


## Export Predictions and Metrics

In [9]:
# Export the 90-day prediction dataset and model metrics for Power BI.

df.to_csv("../data/processed/future_risk_predictions.csv", index=False)

metrics_df = pd.DataFrame([{
    "model": "RandomForest_FutureHighLoss90D",
    "best_threshold": best_threshold_main,
    "roc_auc": roc,
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "f1_score": f1_score(y_test, pred, zero_division=0),
    "test_rows": len(X_test),
    "positive_rate_test": float(y_test.mean())
}])

metrics_df.to_csv("../data/processed/future_risk_model_metrics.csv", index=False)

print("Saved: ../data/processed/future_risk_predictions.csv")
print("Saved: ../data/processed/future_risk_model_metrics.csv")

Saved: ../data/processed/future_risk_predictions.csv
Saved: ../data/processed/future_risk_model_metrics.csv


## Top Predicted Future Risks

In [10]:
# Display the highest-risk forecast records to validate that the model output
# can support a ranked risk monitoring dashboard.

top_risks = (
    df.sort_values("future_high_loss_pred_proba", ascending=False)[[
        "business_line",
        "event_type",
        "causal_factor",
        "province_occurred",
        "year_month",
        "future_high_loss_pred_proba"
    ]]
    .head(25)
    .reset_index(drop=True)
)

display(top_risks)

,business_line,event_type,causal_factor,province_occurred,year_month,future_high_loss_pred_proba
0,Commercial banking,Internal fraud,People,Sichuan,2014-02,0.772116
1,Payment and settlement,Internal fraud,People,Guangdong,2004-02,0.763579
2,Commercial banking,Internal fraud,People,Yunnan,2014-06,0.751239
3,Commercial banking,Internal fraud,People,Beijing,2003-04,0.746927
4,Commercial banking,Internal fraud,People,Hunan,2013-01,0.743687
5,Commercial banking,External fraud,External event,Zhejiang,2013-03,0.730789
6,Commercial banking,Internal fraud,People,Guangdong,2007-12,0.717232
7,Payment and settlement,Internal fraud,People,Shandong,2006-07,0.715819
8,Commercial banking,Internal fraud,People,Henan,2003-11,0.715319
9,Commercial banking,Internal fraud,People,Shanghai,2009-11,0.715319


## Train New Models

In [11]:
# ============================================
# MODEL COMPARISON: RF vs Logistic vs ExtraTrees
# ============================================
# Compare the primary Random Forest model against Logistic Regression and Extra Trees.
# Logistic Regression provides an interpretable baseline, while Extra Trees provides
# another ensemble benchmark.

MODEL_DIR = "../outputs/models"
os.makedirs(MODEL_DIR, exist_ok=True)

model_registry = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

comparison_metrics = []
comparison_predictions = df[[
    "business_line",
    "event_type",
    "causal_factor",
    "province_occurred",
    "year_month",
    target_col
]].copy()

for model_name, clf in model_registry.items():
    comparison_pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", clf)
    ])

    comparison_pipe.fit(X_train, y_train)

    test_proba = comparison_pipe.predict_proba(X_test)[:, 1]
    
    best_threshold, threshold_df = find_best_threshold(y_test, test_proba, metric="f1")
    test_pred = (test_proba >= best_threshold).astype(int)

    roc = roc_auc_score(y_test, test_proba)
    acc = accuracy_score(y_test, test_pred)
    prec = precision_score(y_test, test_pred, zero_division=0)
    rec = recall_score(y_test, test_pred, zero_division=0)
    f1 = f1_score(y_test, test_pred, zero_division=0)
    cm = confusion_matrix(y_test, test_pred)

    tn, fp, fn, tp = cm.ravel()

    comparison_metrics.append({
        "model": model_name,
        "best_threshold": best_threshold,
        "roc_auc": roc,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "test_rows": len(y_test),
        "positive_rate_test": float(y_test.mean())
    })

    # Score full dataset for dashboard comparison
    full_proba = comparison_pipe.predict_proba(X)[:, 1]
    full_pred = (full_proba >= best_threshold).astype(int)

    comparison_predictions[f"{model_name}_pred_proba"] = full_proba
    comparison_predictions[f"{model_name}_pred_flag"] = full_pred

    # Save model artifact
    joblib.dump(comparison_pipe, os.path.join(MODEL_DIR, f"{model_name}_pipeline.joblib"))

comparison_metrics_df = pd.DataFrame(comparison_metrics).sort_values("roc_auc", ascending=False)

comparison_metrics_df.to_csv("../data/processed/model_comparison_metrics.csv", index=False)
comparison_predictions.to_csv("../data/processed/model_comparison_predictions.csv", index=False)

print("Saved: ../data/processed/model_comparison_metrics.csv")
print("Saved: ../data/processed/model_comparison_predictions.csv")
print("Saved model files to:", MODEL_DIR)

display(comparison_metrics_df)
display(comparison_predictions.head())

Saved: ../data/processed/model_comparison_metrics.csv
Saved: ../data/processed/model_comparison_predictions.csv
Saved model files to: ../outputs/models


,model,best_threshold,roc_auc,accuracy,precision,recall,f1_score,true_negatives,false_positives,false_negatives,true_positives,test_rows,positive_rate_test
2,ExtraTrees,0.51,0.785584,0.897092,0.090909,0.4,0.148148,397,40,6,4,447,0.022371
0,RandomForest,0.32,0.738902,0.937360,0.125000,0.3,0.176471,416,21,7,3,447,0.022371
1,LogisticRegression,0.70,0.603432,0.946309,0.111111,0.2,0.142857,421,16,8,2,447,0.022371


,business_line,event_type,causal_factor,province_occurred,year_month,future_high_loss,RandomForest_pred_proba,RandomForest_pred_flag,LogisticRegression_pred_proba,LogisticRegression_pred_flag,ExtraTrees_pred_proba,ExtraTrees_pred_flag
0,Agency services,Business disruption and system failures,System,Guangdong,2008-02,0,0.032841,0,0.424728,0,0.085331,0
1,Agency services,"Clients, products & business practices",People,Beijing,2015-05,0,0.030564,0,0.424464,0,0.063865,0
2,Agency services,"Clients, products & business practices",People,Hainan,2009-10,0,0.115251,0,0.424686,0,0.136117,0
3,Agency services,"Execution, delivery & process management",External event,Shanghai,2010-08,0,0.115362,0,0.424651,0,0.191817,0
4,Agency services,"Execution, delivery & process management",People,Hebei,2003-07,0,0.055863,0,0.424910,0,0.085324,0


## Ranking Format Export

In [12]:
# Reshape model metrics into long format for easier Power BI charting.

metrics_long = comparison_metrics_df.melt(
    id_vars=["model", "test_rows", "positive_rate_test"],
    value_vars=["roc_auc", "accuracy", "precision", "recall", "f1_score"],
    var_name="metric",
    value_name="metric_value"
)

metrics_long.to_csv("../data/processed/model_comparison_metrics_long.csv", index=False)

print("Saved: ../data/processed/model_comparison_metrics_long.csv")
display(metrics_long.head(15))

Saved: ../data/processed/model_comparison_metrics_long.csv


,model,test_rows,positive_rate_test,metric,metric_value
0,ExtraTrees,447,0.022371,roc_auc,0.785584
1,RandomForest,447,0.022371,roc_auc,0.738902
2,LogisticRegression,447,0.022371,roc_auc,0.603432
3,ExtraTrees,447,0.022371,accuracy,0.897092
4,RandomForest,447,0.022371,accuracy,0.937360
5,LogisticRegression,447,0.022371,accuracy,0.946309
6,ExtraTrees,447,0.022371,precision,0.090909
7,RandomForest,447,0.022371,precision,0.125000
8,LogisticRegression,447,0.022371,precision,0.111111
9,ExtraTrees,447,0.022371,recall,0.400000


## 90d vs 180d vs 365d

In [13]:
# ============================================
# HORIZON COMPARISON: 90d vs 180d vs 365d
# ============================================
# Run the same three-model comparison across 90, 180, and 365-day horizons.
# This tests whether model performance changes across short-, medium-, and long-term forecasting windows.

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

horizon_files = {
    90: "../data/processed/future_model_dataset_90d.csv",
    180: "../data/processed/future_model_dataset_180d.csv",
    365: "../data/processed/future_model_dataset_365d.csv"
}

feature_cols = [
    "business_line",
    "event_type",
    "causal_factor",
    "province_occurred",
    "event_count",
    "total_loss_cny",
    "avg_loss_cny",
    "median_loss_cny",
    "max_loss_cny",
    "high_loss_count",
    "avg_banks_involved",
    "event_count_lag_1",
    "event_count_lag_2",
    "event_count_lag_3",
    "event_count_lag_6",
    "event_count_lag_12",
    "total_loss_lag_1",
    "total_loss_lag_2",
    "total_loss_lag_3",
    "total_loss_lag_6",
    "total_loss_lag_12",
    "high_loss_count_lag_1",
    "high_loss_count_lag_2",
    "high_loss_count_lag_3",
    "high_loss_count_lag_6",
    "high_loss_count_lag_12",
    "event_count_roll_3",
    "event_count_roll_6",
    "event_count_roll_12",
    "total_loss_roll_3",
    "total_loss_roll_6",
    "total_loss_roll_12",
    "high_loss_roll_3",
    "high_loss_roll_6",
    "high_loss_roll_12",
    "loss_per_event",
    "log_total_loss",
    "business_line_freq",
    "event_type_freq",
    "causal_factor_freq",
    "province_occurred_freq",
    "month_num",
    "quarter_num",
    "year_num"
]

model_registry = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

horizon_comparison_rows = []

for horizon_days, file_path in horizon_files.items():
    horizon_df = pd.read_csv(file_path).copy()
    horizon_df["future_high_loss"] = horizon_df["future_high_loss"].astype(int)

    X_h = horizon_df[feature_cols].copy()
    y_h = horizon_df["future_high_loss"].copy()

    X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
        X_h, y_h,
        test_size=0.2,
        random_state=42,
        stratify=y_h
    )

    for model_name, clf in model_registry.items():
        horizon_pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", clf)
        ])

        horizon_pipe.fit(X_train_h, y_train_h)
        test_proba_h = horizon_pipe.predict_proba(X_test_h)[:, 1]

        best_threshold_h, _ = find_best_threshold(y_test_h, test_proba_h, metric="f1")
        test_pred_h = (test_proba_h >= best_threshold_h).astype(int)

        roc_h = roc_auc_score(y_test_h, test_proba_h)
        acc_h = accuracy_score(y_test_h, test_pred_h)
        prec_h = precision_score(y_test_h, test_pred_h, zero_division=0)
        rec_h = recall_score(y_test_h, test_pred_h, zero_division=0)
        f1_h = f1_score(y_test_h, test_pred_h, zero_division=0)
        cm_h = confusion_matrix(y_test_h, test_pred_h)
        tn_h, fp_h, fn_h, tp_h = cm_h.ravel()

        horizon_comparison_rows.append({
            "horizon_days": horizon_days,
            "model": model_name,
            "best_threshold": best_threshold_h,
            "roc_auc": roc_h,
            "accuracy": acc_h,
            "precision": prec_h,
            "recall": rec_h,
            "f1_score": f1_h,
            "true_negatives": tn_h,
            "false_positives": fp_h,
            "false_negatives": fn_h,
            "true_positives": tp_h,
            "test_rows": len(y_test_h),
            "positive_rate_test": float(y_test_h.mean())
        })

horizon_comparison_df = pd.DataFrame(horizon_comparison_rows).sort_values(
    ["horizon_days", "roc_auc"], ascending=[True, False]
)

horizon_comparison_df.to_csv("../data/processed/horizon_model_comparison_metrics.csv", index=False)

print("Saved: ../data/processed/horizon_model_comparison_metrics.csv")
display(horizon_comparison_df)

Saved: ../data/processed/horizon_model_comparison_metrics.csv


,horizon_days,model,best_threshold,roc_auc,accuracy,precision,recall,f1_score,true_negatives,false_positives,false_negatives,true_positives,test_rows,positive_rate_test
2,90,ExtraTrees,0.51,0.785584,0.897092,0.090909,0.400000,0.148148,397,40,6,4,447,0.022371
0,90,RandomForest,0.32,0.738902,0.937360,0.125000,0.300000,0.176471,416,21,7,3,447,0.022371
1,90,LogisticRegression,0.70,0.603432,0.946309,0.111111,0.200000,0.142857,421,16,8,2,447,0.022371
5,180,ExtraTrees,0.57,0.876845,0.903803,0.222222,0.555556,0.317460,394,35,8,10,447,0.040268
3,180,RandomForest,0.27,0.867910,0.829978,0.162791,0.777778,0.269231,357,72,4,14,447,0.040268
4,180,LogisticRegression,0.48,0.640896,0.870246,0.142857,0.444444,0.216216,381,48,10,8,447,0.040268
8,365,ExtraTrees,0.64,0.826307,0.897092,0.375000,0.529412,0.439024,383,30,16,18,447,0.076063
6,365,RandomForest,0.50,0.806367,0.894855,0.341463,0.411765,0.373333,386,27,20,14,447,0.076063
7,365,LogisticRegression,0.48,0.755234,0.859060,0.254237,0.441176,0.322581,369,44,19,15,447,0.076063
